# Pandas Data Engineering Project

This notebook contains practical Data Engineering scenarios using a single business dataset.  
The exercises cover data ingestion, profiling, filtering, cleaning, transformation, aggregation, dates, ranking, reshaping, validation, and ETL-style processing.

**Domain:** Food_Delivery_Analytics

**Input file:** `06_Food_Delivery.csv`

**Questions:** 30

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

df = pd.read_csv("06_Food_Delivery.csv")
df.head()


## Dataset Information

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isna().sum())


## Q1. Numeric summary

Create count, mean, minimum and maximum for the main numeric measure.

In [ ]:
print(df["order_value"].agg(["count","mean","min","max"]))

## Q2. High-value records

Identify records above the 75th percentile of order_value.

In [ ]:
t=    order_id customer_id      restaurant  ... rating order_status  order_date
0    FD00001       C0024     Food Corner  ...    2.0    Delivered  2025-02-04
1    FD00002       C0025  Tandoori House  ...    4.1    Delivered  2025-04-21
2    FD00003       C0177  Tandoori House  ...    3.6    Delivered  2025-03-31
3    FD00004       C0162       Spice Hub  ...    2.9    Delivered  2025-05-06
4    FD00005       C0031  Tandoori House  ...    2.3    Delivered  2025-11-08
..       ...         ...             ...  ...    ...          ...         ...
397  FD00398       C0148      Green Bowl  ...    1.0    Delivered  2025-07-09
398  FD00399       C0117     Urban Bites  ...    3.1    Delivered  2026-05-20
399  FD00400       C0052      Green Bowl  ...    4.4    Delivered  2025-11-10
400  FD00008       C0172         Cafe 24  ...    4.0    Delivered  2026-05-15
401  FD00036       C0211     Food Corner  ...    4.1    Delivered  2026-08-06

[402 rows x 11 columns]
threshold=df["order_value"].quantile(.75)
print(df[df["order_value"]>threshold].sort_values("order_value",ascending=False).head(20))

## Q3. Category aggregation

Calculate record count, total order_value, and average delivery_fee for each order_id.

In [ ]:
print(df.groupby("order_id",dropna=False).agg(count=("order_value","count"),total=("order_value","sum"),avg=("delivery_fee","mean")).sort_values("total",ascending=False))

## Q4. Two-dimensional aggregation

Analyze order_value by order_id and customer_id; keep groups with at least 5 records.

In [ ]:
r=df.groupby(["order_id","customer_id"],dropna=False).agg(count=("order_value","count"),total=("order_value","sum")).query("count>=5").sort_values("total",ascending=False); print(r)

## Q5. Ranking

Rank records by the main numeric measure and display the top 15.

In [ ]:
x=df.copy(); x["rank"]=x["order_value"].rank(method="dense",ascending=False); print(x.sort_values("rank").head(15))

## Q6. Duplicate audit

Identify duplicate records using the first identifier column.

In [ ]:
key="order_id"; print(df[df.duplicated(key,keep=False)].sort_values(key))

## Q7. Duplicate removal

Create a deduplicated dataset using the first identifier column.

In [ ]:
x=df.drop_duplicates("order_id"); print("Before:",len(df),"After:",len(x))

## Q8. Missing-value profile

Calculate null percentage for every column and sort descending.

In [ ]:
print((df.isna().mean()*100).round(2).sort_values(ascending=False))

## Q9. Categorical fill

Fill missing values in a categorical column with Unknown.

In [ ]:
x=df.copy(); x["order_id"]=x["order_id"].fillna("Unknown"); print(x["order_id"].value_counts(dropna=False))

## Q10. Numeric median fill

Fill missing numeric values using the overall median.

In [ ]:
x=df.copy(); x["order_value"]=x["order_value"].fillna(x["order_value"].median()); print(x["order_value"].isna().sum())

## Q11. String search

Find records where order_id contains the letter a, case-insensitively.

In [ ]:
print(df[df["order_id"].astype("string").str.contains("a",case=False,na=False)].head(30))

## Q12. Sorted analysis

Sort the dataset by order_value descending and delivery_fee ascending.

In [ ]:
print(df.sort_values(["order_value","delivery_fee"],ascending=[False,True]).head(25))

## Q13. Conditional segmentation

Create a new segment based on order_value using its median as the threshold.

In [ ]:
x=df.copy(); med=x["order_value"].median(); x["segment"]=np.where(x["order_value"]>=med,"Above Median","Below Median"); print(x["segment"].value_counts())

## Q14. Transform

Use groupby transform to calculate each record's group average of the main measure.

In [ ]:
x=df.copy(); x["group_avg"]=x.groupby("order_id")["order_value"].transform("mean"); print(x[["order_id","order_value","group_avg"]].head(25))

## Q15. Group share

Calculate each order_id's share of total order_value.

In [ ]:
g=df.groupby("order_id",dropna=False)["order_value"].sum(); print((g/g.sum()*100).round(2).sort_values(ascending=False))

## Q16. Pivot report

Create a pivot table showing average order_value by order_id and customer_id.

In [ ]:
print(pd.pivot_table(df,index="order_id",columns="customer_id",values="order_value",aggfunc="mean"))

## Q17. Melt report

Create a compact wide summary and convert it to long format using melt.

In [ ]:
w=df.groupby("order_id")[["order_value","delivery_fee"]].mean().reset_index(); print(w.melt(id_vars="order_id",var_name="metric",value_name="value"))

## Q18. Validation rule

Count records where order_value is negative or missing.

In [ ]:
print("Invalid:",df["order_value"].isna().sum()+(df["order_value"]<0).sum())

## Q19. Outlier detection

Use IQR to identify high outliers in order_value.

In [ ]:
q1=df["order_value"].quantile(.25); q3=df["order_value"].quantile(.75); iqr=q3-q1; print(df[df["order_value"]>q3+1.5*iqr][["order_value"]].head(30))

## Q20. Top group

Find the category in order_id with the highest total order_value.

In [ ]:
print(df.groupby("order_id")["order_value"].sum().nlargest(1))

## Q21. Bottom group

Find the three categories in order_id with the lowest average order_value.

In [ ]:
print(df.groupby("order_id")["order_value"].mean().nsmallest(3))

## Q22. Multiple aggregations

Produce min, max, mean and median of order_value for each order_id.

In [ ]:
print(df.groupby("order_id")["order_value"].agg(["min","max","mean","median"]))

## Q23. Apply rule

Use apply to classify order_value relative to its median.

In [ ]:
x=df.copy(); med=x["order_value"].median(); x["classification"]=x["order_value"].apply(lambda v:"High" if pd.notna(v) and v>=med else "Low"); print(x["classification"].value_counts())

## Q24. Map categories

Create a numeric code for the most frequent categories in the first categorical field.

In [ ]:
codes={{v:i for i,v in enumerate(df["order_id"].dropna().unique())}}; x=df.copy(); x["order_id_code"]=x["order_id"].map(codes); print(x[["order_id","order_id_code"]].head(20))

## Q25. ETL filtering

Build an ETL subset containing non-null order_value, non-null order_id, and values above the median order_value.

In [ ]:
m=df["order_value"].median(); x=df[df["order_value"].notna()&df["order_id"].notna()&(df["order_value"]>m)].copy(); print(x.shape); print(x.head())

## Q26. Chunk processing

Read the input CSV in chunks of 50 rows and calculate the main numeric total.

In [ ]:
total=0
count=0
for chunk in pd.read_csv("06_Food_Delivery.csv",chunksize=50):
    total += chunk["order_value"].sum()
    count += len(chunk)
print("Rows processed:",count)
print("Total:",total)

## Q27. Final report

Create a final order_id-level report with count, total order_value, average delivery_fee, and rank by total order_value.

In [ ]:
r=df.groupby("order_id",dropna=False).agg(records=("order_value","count"),total=("order_value","sum"),average=("delivery_fee","mean")).sort_values("total",ascending=False); r["rank"]=r["total"].rank(method="dense",ascending=False).astype(int); print(r)

## End of Project

The notebook is ready to execute with the supplied input CSV.